## 01 — Style Functions

In the last lesson every feature on the map looked the same. In this lesson we change that.

A **style function** is a Python function you pass to `GeoJSON()`. ipyleaflet calls it once per feature, passing the feature dict as the argument. Whatever style dict you return determines how that feature is drawn.

```
feature  →  style function  →  style dict  →  visual appearance
```

This is the moment where a map becomes a **data-driven visualization**.

## Setup

In [ ]:
import json
from ipyleaflet import Map, GeoJSON

WICHITA_FALLS = (33.9137, -98.4934)

with open("../02-Viewing_GeoJSON/data/wichita_falls.geojson") as f:
    geojson = json.load(f)

## What is a Style Function?

The `GeoJSON` class accepts a `style` argument. Pass it a function that takes one feature and returns a dict.

The simplest possible style function returns the same dict for every feature:

In [ ]:
def uniform_style(feature):
    return {
        "color":       "#e74c3c",   # border / stroke color
        "weight":      2,            # border thickness (px)
        "fillColor":   "#e74c3c",   # polygon fill color
        "fillOpacity": 0.4,          # polygon fill opacity (0–1)
    }

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=geojson, style=uniform_style))
m

### Style dict keys

These are the keys ipyleaflet recognizes (they come from Leaflet.js):

| Key | What it affects | Example value |
|---|---|---|
| `color` | stroke (border) color | `"#e74c3c"`, `"red"` |
| `weight` | stroke thickness in px | `2` |
| `opacity` | stroke opacity | `0.8` |
| `fillColor` | polygon/point fill color | `"#3498db"` |
| `fillOpacity` | polygon/point fill opacity | `0.5` |
| `dashArray` | dashed stroke pattern | `"4 4"` |

Not every key applies to every geometry type — `fillColor` has no effect on a `LineString`, and `dashArray` will not affect a `Point`.

## Conditional Styling

The real power is using `if` statements inside the style function to return different styles based on a feature's properties.

Our dataset has a `type` property with values like `"park"`, `"education"`, `"water"`, etc. Let's color each type differently.

In [ ]:
def style_by_type(feature):
    feat_type = feature["properties"].get("type", "unknown")

    if feat_type == "park":
        return {"color": "#2ecc71", "fillColor": "#2ecc71", "fillOpacity": 0.4, "weight": 2}
    elif feat_type == "water":
        return {"color": "#1abc9c", "fillColor": "#1abc9c", "fillOpacity": 0.4, "weight": 2}
    elif feat_type == "education":
        return {"color": "#3498db", "fillColor": "#3498db", "fillOpacity": 0.4, "weight": 2}
    elif feat_type == "government":
        return {"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 0.4, "weight": 2}
    elif feat_type == "route":
        return {"color": "#e67e22", "weight": 3, "opacity": 0.8}
    else:
        return {"color": "#95a5a6", "fillColor": "#95a5a6", "fillOpacity": 0.3, "weight": 1}

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=geojson, style=style_by_type))
m

Each feature now has a distinct color based on its `type`. The logic is entirely in Python — ipyleaflet just calls the function.

## Refactoring: a Color Lookup Dict

Long chains of `if/elif` for color lookup get messy. A dict is cleaner — look up the color, fall back to a default if the key is missing.

In [ ]:
COLOR_MAP = {
    "park":       "#2ecc71",
    "water":      "#1abc9c",
    "education":  "#3498db",
    "government": "#e74c3c",
    "route":      "#e67e22",
}

def style_by_type(feature):
    feat_type = feature["properties"].get("type", "unknown")
    color = COLOR_MAP.get(feat_type, "#95a5a6")  # default: gray
    return {"color": color, "fillColor": color, "fillOpacity": 0.4, "weight": 2}

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=geojson, style=style_by_type))
m

Same result, less code. Adding a new category means adding one line to `COLOR_MAP`.

## Numeric Styling and Data Classification

When a property holds a **number**, you can classify features into buckets and assign a color to each bucket. This is called **data classification** — the same technique used in choropleth maps.

Our `wichita_falls.geojson` does not have numeric properties, so here is a small inline example. The pattern is identical regardless of dataset.

In [ ]:
# A small dataset with a numeric "score" property
scored = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.49, 33.91]}, "properties": {"name": "Site A", "score": 91}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.51, 33.89]}, "properties": {"name": "Site B", "score": 55}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.53, 33.87]}, "properties": {"name": "Site C", "score": 22}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.50, 33.93]}, "properties": {"name": "Site D", "score": 78}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.47, 33.88]}, "properties": {"name": "Site E", "score": 10}},
    ]
}

def style_by_score(feature):
    score = feature["properties"].get("score", 0)

    if score >= 75:
        color = "#e74c3c"   # red — high score
    elif score >= 40:
        color = "#e67e22"   # orange — medium score
    else:
        color = "#3498db"   # blue — low score

    return {"color": color, "fillColor": color, "fillOpacity": 0.7, "weight": 1}

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=scored, style=style_by_score))
m

The classification thresholds (`>= 75`, `>= 40`) are your design decision. Common approaches:

- **Equal intervals** — divide the data range into equal-width buckets
- **Quantiles** — divide so each bucket has the same number of features
- **Natural breaks** — cluster around natural groupings in the data
- **Manual thresholds** — domain knowledge (e.g., above/below a meaningful value)

For now, manual thresholds are fine.

## Styling Multiple Attributes Together

You are not limited to one key per style dict. Return as many as you need — and they can all vary per feature.

In [ ]:
def detailed_style(feature):
    feat_type = feature["properties"].get("type", "unknown")
    color = COLOR_MAP.get(feat_type, "#95a5a6")

    # Routes get a dashed line and heavier weight; everything else is solid
    if feat_type == "route":
        return {
            "color":     color,
            "weight":    4,
            "opacity":   0.9,
            "dashArray": "6 4",
        }
    else:
        return {
            "color":       color,
            "weight":      2,
            "fillColor":   color,
            "fillOpacity": 0.45,
        }

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=geojson, style=detailed_style))
m

## Lambda Style Functions

When the style logic is a single expression, you can replace a full `def` with a **lambda**. A lambda is an anonymous one-liner function.

```python
# These two are equivalent:
def style(feature):
    return {"color": "blue"}

style = lambda feature: {"color": "blue"}
```

Lambdas are useful for quick one-off styles — especially when you are exploring data and do not want to name a function.

In [ ]:
# Uniform style as a lambda — no def needed
m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(
    data=geojson,
    style=lambda feature: {"color": "#8e44ad", "fillColor": "#8e44ad", "fillOpacity": 0.3, "weight": 2}
))
m

In [ ]:
# Lambda with a ternary — still one expression
m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(
    data=geojson,
    style=lambda f: (
        {"color": "#2ecc71", "fillColor": "#2ecc71", "fillOpacity": 0.5, "weight": 2}
        if f["properties"].get("type") == "park"
        else {"color": "#95a5a6", "fillColor": "#95a5a6", "fillOpacity": 0.3, "weight": 1}
    )
))
m

**When to use a lambda vs a `def`:**

| Situation | Use |
|---|---|
| Uniform style, exploring data | lambda |
| One simple condition | lambda with ternary |
| Multiple conditions or a color map | `def` |
| Style function used in more than one place | `def` |

When in doubt, use `def` — it is easier to read and debug.

## Exercise A

Write a style function that colors features by **geometry type** rather than by the `type` property:

| Geometry | Color |
|---|---|
| `Point` | red `#e74c3c` |
| `LineString` | orange `#e67e22` |
| `Polygon` | blue `#2980b9` |

Display `wichita_falls.geojson` with this function.

In [ ]:
from ipyleaflet import Map, GeoJSON

# Style by geometry type: Points=red, LineStrings=orange, Polygons=blue
def style_by_geom(feature):
    # Your code here
    pass

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=geojson, style=style_by_geom))
m

## Exercise C

Extend the numeric style function to use **4 color levels** instead of 3:

| Score | Color |
|---|---|
| `>= 75` | red `#e74c3c` |
| `50–74` | orange `#e67e22` |
| `25–49` | yellow `#f1c40f` |
| `< 25` | blue `#3498db` |

Apply it to the `scored` dataset from the teaching cells.

In [ ]:
def style_by_score_4(feature):
    """4-level color scale: < 25, 25–49, 50–74, >= 75"""
    # Your code here
    pass

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=scored, style=style_by_score_4))
m

---

## Check Your Understanding

Write a style function for `geojson` that:

- Colors `"water"` and `"park"` features **green** (`#27ae60`)
- Colors everything else **gray** (`#7f8c8d`)
- Sets `fillOpacity` to `0.5` and `weight` to `2` for all features

Display the result on a map.

```python
# your answer here
```

## Next

In [02 — Filtering](./02-Filtering.ipynb), we control which features appear on the map at all — combining filtering and styling into a full data-driven pipeline.